In [1]:
"""
Step 8: Direct viability comparison of the 3 architectures:
  1. Retuned Boxcar (bx)        : 1 rectangular window [a, b), 2 features/qubit
  2. Old Dual-Filter (df)       : 2 BRAMs + 2 MACs per qubit (WMF + WRMF), 4 features/qubit + 4-cycle normalizer
  3. New Split-Latch (sf_split) : 1 BRAM + 1 MAC per qubit (WMF latched at T_mid and T_end), 4 features/qubit
                                  + simplified 1-cycle shift normalizer (or 0-cycle wire-slice in linear mode)
"""
import os
import numpy as np

# ------------------------------------------------------------------ config
DATA_DIR = r"D:\MSU\Qubit_readout_FPGA\python"
SEED = 42
POOL_TRAIN_EACH = 100_000
N_SAMPLES = 770

N_QUBITS = 5
FS = 500.0e6
IF_FREQS = np.array([64.729e6, 25.366e6, -24.790e6, -70.269e6, -127.282e6])
T_BINS = 500
T_MID = 225                        # Mid-window latch bin for the single-filter split accumulator
EMA_ALPHA = 0.05
RIDGE_LAMBDA_REL = 1e-4
CLIP = 99.99
N_CAL, N_CLF, N_TEST = 30_000, 50_000, 100_000
STRENGTHS = [0.0, 1.0, 2.0]


def find_file(fname):
    for d in [DATA_DIR, os.path.dirname(DATA_DIR), "."]:
        cand = os.path.join(d, fname)
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(f"Cannot find {fname}")


# ------------------------------------------------------------------ load pools
Xtr = np.load(find_file("0528_X_train_0_770.npy"), mmap_mode="r")
Xte_m = np.load(find_file("0528_X_test_0_770.npy"), mmap_mode="r")
ytr = np.load(find_file("0528_y_train_0_770.npy")).astype(int)
yte = np.load(find_file("0528_y_test_0_770.npy")).astype(int)

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(ytr))
cal_pool = np.sort(perm[:POOL_TRAIN_EACH])
clf_pool = np.sort(perm[POOL_TRAIN_EACH:2 * POOL_TRAIN_EACH])


def load_pool(X, idx):
    out = np.empty((len(idx), 2 * N_SAMPLES), np.int16)
    for i in range(0, len(idx), 20_000):
        out[i:i + 20_000] = np.rint(X[idx[i:i + 20_000]]).astype(np.int16)
    return out


print("loading source-shot pools ...")
S_cal, y_cal_pool = load_pool(Xtr, cal_pool), ytr[cal_pool]
S_clf, y_clf_pool = load_pool(Xtr, clf_pool), ytr[clf_pool]
S_te, y_te_pool = load_pool(Xte_m, np.arange(len(yte))), yte


def best_threshold(s, y):
    order = np.argsort(s)
    ss, ys = s[order], y[order]
    n0 = (ys == 0).sum()
    c1 = np.r_[0, np.cumsum(ys == 1)]
    c0 = np.r_[0, np.cumsum(ys == 0)]
    k = int(np.argmin(c1 + (n0 - c0)))
    if k == 0:
        return ss[0] - 1.0
    if k == len(ss):
        return ss[-1]
    return 0.5 * (ss[k - 1] + ss[k])


def fid_pred(pred, y):
    return 0.5 * (np.mean(~pred[y == 0]) + np.mean(pred[y == 1]))


def lda_fit(F, y):
    m0, m1 = F[y == 0].mean(0), F[y == 1].mean(0)
    Sw = 0.5 * (np.cov(F[y == 0].T) + np.cov(F[y == 1].T))
    return np.linalg.solve(Sw, m1 - m0)


def ema_1d(x, alpha):
    out = np.empty_like(x)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = alpha * x[t] + (1.0 - alpha) * out[t - 1]
    return out


# ================================================================== PART 1: 5-Qubit Multiplexed Bus Comparison
print("\n################ PART 1: 5-Qubit Bus (Boxcar vs Old Dual-Filter vs New Split-Latch) ################")
idx_cal = rng.integers(0, len(y_cal_pool), size=(N_CAL, N_QUBITS))
idx_clf = rng.integers(0, len(y_clf_pool), size=(N_CLF, N_QUBITS))
idx_te  = rng.integers(0, len(y_te_pool),  size=(N_TEST, N_QUBITS))
Y_cal, Y_clf, Y_te = y_cal_pool[idx_cal], y_clf_pool[idx_clf], y_te_pool[idx_te]

t_sec = np.arange(T_BINS, dtype=np.float64) / FS
carriers = np.exp(1j * 2.0 * np.pi * IF_FREQS[:, None] * t_sec[None, :])
LIN = np.array([[1.00, 0.15, 0.04, 0.01, 0.00], [0.15, 1.00, 0.16, 0.04, 0.01], [0.04, 0.16, 1.00, 0.14, 0.04],
                [0.01, 0.04, 0.14, 1.00, 0.15], [0.00, 0.01, 0.04, 0.15, 1.00]])
PH  = np.array([[0.00, 0.28, 0.08, 0.00, 0.00], [-0.26, 0.00, 0.30, 0.08, 0.00], [0.07, -0.28, 0.00, 0.26, 0.07],
                [0.00, 0.07, -0.25, 0.00, 0.28], [0.00, 0.00, 0.07, -0.27, 0.00]])


def synth(S, idx, Y, s, chunk=5000):
    n = len(idx)
    L = np.eye(N_QUBITS) + s * (LIN - np.eye(N_QUBITS))
    P = s * PH
    I_out = np.empty((n, T_BINS), np.int16)
    Q_out = np.empty((n, T_BINS), np.int16)
    for a in range(0, n, chunk):
        b = min(a + chunk, n)
        bb = np.empty((b - a, N_QUBITS, T_BINS), np.complex128)
        for q in range(N_QUBITS):
            rows = S[idx[a:b, q], :2 * T_BINS].astype(np.float64)
            bb[:, q, :] = rows[:, 0::2] + 1j * rows[:, 1::2]
        sgn = 2.0 * Y[a:b] - 1.0
        theta = sgn @ P.T
        prev = np.zeros_like(sgn)
        prev[:, 1:] = sgn[:, :-1]
        bb = bb * ((1.0 + 0.12 * s * prev)[:, :, None] * np.exp(1j * theta[:, :, None]))
        bb = np.einsum("pq,bqt->bpt", L, bb)
        bus = np.einsum("bkt,kt->bt", bb, carriers) / np.sqrt(N_QUBITS)
        I_out[a:b] = np.clip(np.rint(bus.real), -32768, 32767).astype(np.int16)
        Q_out[a:b] = np.clip(np.rint(bus.imag), -32768, 32767).astype(np.int16)
    return I_out, Q_out


def premix(q, w0I, w0Q, w1I, w1Q, W):
    c, sn = np.cos(2.0 * np.pi * IF_FREQS[q] * t_sec), np.sin(2.0 * np.pi * IF_FREQS[q] * t_sec)
    W[q, 0, 0], W[q, 0, 1] =  w0I * c, w0I * sn
    W[q, 1, 0], W[q, 1, 1] = -w0Q * sn, w0Q * c
    W[q, 2, 0], W[q, 2, 1] =  w1I * c, w1I * sn
    W[q, 3, 0], W[q, 3, 1] = -w1Q * sn, w1Q * c


def build_all_tables(I_raw, Q_raw, Y, a_bx, b_bx, t_mid=T_MID):
    """
    Returns three Stage 2 tables:
      W_bx : Retuned Boxcar [a_bx, b_bx)
      W_df : Old Dual-Filter (WMF + WRMF -> needs 2 BRAMs & 2 MACs per qubit)
      W_sf : New Split-Latch (ONLY WMF, split at t_mid into Early [0, t_mid) and Late [t_mid, T_BINS) -> 1 BRAM & 1 MAC!)
    """
    S_raw = I_raw.astype(np.float64) + 1j * Q_raw.astype(np.float64)
    W_bx = np.zeros((N_QUBITS, 4, 2, T_BINS))
    W_df = np.zeros((N_QUBITS, 4, 2, T_BINS))
    W_sf = np.zeros((N_QUBITS, 4, 2, T_BINS))

    rect = np.zeros(T_BINS)
    rect[a_bx:b_bx] = 1.0
    z = np.zeros(T_BINS)

    for q in range(N_QUBITS):
        premix(q, rect, rect, z, z, W_bx)

        Zq = S_raw * np.exp(-1j * 2.0 * np.pi * IF_FREQS[q] * t_sec)[None, :]
        Iq, Qq, yq = Zq.real, Zq.imag, Y[:, q]
        m0, m1 = yq == 0, yq == 1
        mI0, mQ0, vI0, vQ0 = Iq[m0].mean(0), Qq[m0].mean(0), Iq[m0].var(0), Qq[m0].var(0)
        mI1, mQ1, vI1, vQ1 = Iq[m1].mean(0), Qq[m1].mean(0), Iq[m1].var(0), Qq[m1].var(0)

        # State Matched Filter (WMF)
        wmI = ema_1d(mI1 - mI0, EMA_ALPHA) / (0.5 * (vI0 + vI1))
        wmQ = ema_1d(mQ1 - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQ1))

        # Old RMF (WRMF)
        mtv = np.c_[Iq.mean(1), Qq.mean(1)]
        C0, C1 = mtv[m0].mean(0), mtv[m1].mean(0)
        relax = m1 & (np.linalg.norm(mtv - C0, axis=1) < 0.5 * np.linalg.norm(C0 - C1))
        if relax.sum() >= 20:
            mIR, mQR, vIR, vQR = Iq[relax].mean(0), Qq[relax].mean(0), Iq[relax].var(0), Qq[relax].var(0)
            wrI = ema_1d(mIR - mI0, EMA_ALPHA) / (0.5 * (vI0 + vIR))
            wrQ = ema_1d(mQR - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQR))
        else:
            wrI = wrQ = np.zeros(T_BINS)
        premix(q, wmI, wmQ, wrI, wrQ, W_df)

        # New Split-Latch (1 single WMF table latched at t_mid: Early=[0:t_mid), Late=[t_mid:T_BINS))
        wmI_early, wmQ_early = np.copy(wmI), np.copy(wmQ)
        wmI_early[t_mid:] = 0.0
        wmQ_early[t_mid:] = 0.0
        wmI_late, wmQ_late = np.copy(wmI), np.copy(wmQ)
        wmI_late[:t_mid] = 0.0
        wmQ_late[:t_mid] = 0.0
        premix(q, wmI_early, wmQ_early, wmI_late, wmQ_late, W_sf)

    return W_bx, W_df, W_sf


def accumulate(I_raw, Q_raw, W, chunk=5000):
    N = len(I_raw)
    out = np.empty((N, N_QUBITS, 4))
    WI = W[:, :, 0, :].reshape(N_QUBITS * 4, T_BINS).T
    WQ = W[:, :, 1, :].reshape(N_QUBITS * 4, T_BINS).T
    for s_ in range(0, N, chunk):
        e = min(s_ + chunk, N)
        out[s_:e] = (I_raw[s_:e].astype(np.float64) @ WI + Q_raw[s_:e].astype(np.float64) @ WQ).reshape(e - s_, N_QUBITS, 4)
    return out


def normalize(Ac, At):
    mu = Ac.mean(0)
    A = np.percentile(np.abs(Ac - mu), CLIP, axis=0)
    n = np.ceil(np.log2(np.maximum(A, 1e-9)))
    f = lambda a: np.clip((a + 2.0 ** n - mu) / 2.0 ** (n + 1), 0, 1) - 0.5
    return f(Ac), f(At)


def quad_cross(V, q, nf):
    vk = V[:, q, :nf]
    self_q = [vk[:, i] * vk[:, j] for i in range(nf) for j in range(i, nf)]
    cross = []
    for j in range(N_QUBITS):
        if j != q:
            vj = V[:, j, :]
            # For split-latch, (vj[:,0]+vj[:,2]) is the full WMF I-sum and (vj[:,1]+vj[:,3]) is the full WMF Q-sum
            if V is not None and nf == 4 and hasattr(V, "_is_sf"):
                ak, bk = vk[:, 0] + vk[:, 2], vk[:, 1] + vk[:, 3]
                aj, bj = vj[:, 0] + vj[:, 2], vj[:, 1] + vj[:, 3]
            else:
                ak, bk = vk[:, 0], vk[:, 1]
                aj, bj = vj[:, 0], vj[:, 1]
            cross += [ak * aj, ak * bj, bk * aj, bk * bj]
    return np.stack(self_q, 1), np.stack(cross, 1)


def get_feats(arch, Vbx, Vdf, Vsf, q):
    if arch == "bx_single":
        return Vbx[:, q, :2]
    if arch == "bx_lin10":
        return Vbx[:, :, :2].reshape(len(Vbx), -1)
    if arch == "bx_cross":
        sq, cr = quad_cross(Vbx, q, 2)
        return np.c_[Vbx[:, :, :2].reshape(len(Vbx), -1), sq, cr]

    if arch == "df_single":
        return Vdf[:, q, :]
    if arch == "df_lin20":
        return Vdf.reshape(len(Vdf), -1)
    if arch == "df_cross":
        sq, cr = quad_cross(Vdf, q, 4)
        return np.c_[Vdf.reshape(len(Vdf), -1), sq, cr]

    if arch == "sf_single":
        return Vsf[:, q, :]
    if arch == "sf_lin20":
        return Vsf.reshape(len(Vsf), -1)
    if arch == "sf_self30":
        # 20 linear features + 10 intra-qubit quadratic products (captures T1 jump boundary, 0 neighbor cross-mults)
        vk = Vsf[:, q, :]
        sq = np.stack([vk[:, i] * vk[:, j] for i in range(4) for j in range(i, 4)], 1)
        return np.c_[Vsf.reshape(len(Vsf), -1), sq]
    if arch == "sf_cross":
        vk = Vsf[:, q, :]
        sq = np.stack([vk[:, i] * vk[:, j] for i in range(4) for j in range(i, 4)], 1)
        ak, bk = vk[:, 0] + vk[:, 2], vk[:, 1] + vk[:, 3]
        cross = []
        for j in range(N_QUBITS):
            if j != q:
                vj = Vsf[:, j, :]
                aj, bj = vj[:, 0] + vj[:, 2], vj[:, 1] + vj[:, 3]
                cross += [ak * aj, ak * bj, bk * aj, bk * bj]
        return np.c_[Vsf.reshape(len(Vsf), -1), sq, np.stack(cross, 1)]
    raise ValueError(arch)


ARCHS = [
    "bx_single", "bx_lin10", "bx_cross",
    "df_single", "df_lin20", "df_cross",
    "sf_single", "sf_lin20", "sf_self30", "sf_cross",
]
TUNE_WINDOWS = {0.0: (75, 500), 1.0: (100, 500), 2.0: (125, 500)}  # Optimal windows found in Step 7 Check A

for s in STRENGTHS:
    a_best, b_best = TUNE_WINDOWS[s]
    print(f"\n=== Crosstalk strength s = {s} (Boxcar window [{a_best}, {b_best}), Split-Latch T_mid = {T_MID}) ===")
    I_cal, Q_cal = synth(S_cal, idx_cal, Y_cal, s)
    I_clf, Q_clf = synth(S_clf, idx_clf, Y_clf, s)
    I_te,  Q_te  = synth(S_te,  idx_te,  Y_te,  s)

    W_bx, W_df, W_sf = build_all_tables(I_cal, Q_cal, Y_cal, a_best, b_best, T_MID)
    Vbx_c, Vbx_t = normalize(accumulate(I_clf, Q_clf, W_bx), accumulate(I_te, Q_te, W_bx))
    Vdf_c, Vdf_t = normalize(accumulate(I_clf, Q_clf, W_df), accumulate(I_te, Q_te, W_df))
    Vsf_c, Vsf_t = normalize(accumulate(I_clf, Q_clf, W_sf), accumulate(I_te, Q_te, W_sf))

    corr, fids, n_mults = {}, {}, {}
    for arch in ARCHS:
        corr[arch] = np.zeros(Y_te.shape, bool)
        fids[arch] = []
        for q in range(N_QUBITS):
            Fc = np.c_[np.ones(len(Vdf_c)), get_feats(arch, Vbx_c, Vdf_c, Vsf_c, q)]
            Ft = np.c_[np.ones(len(Vdf_t)), get_feats(arch, Vbx_t, Vdf_t, Vsf_t, q)]
            n_mults[arch] = Fc.shape[1]
            w = np.linalg.solve(Fc.T @ Fc + RIDGE_LAMBDA_REL * len(Fc) * np.eye(Fc.shape[1]),
                                Fc.T @ (2.0 * Y_clf[:, q] - 1.0))
            pq = (Ft @ w) > best_threshold(Fc @ w, Y_clf[:, q])
            corr[arch][:, q] = pq == (Y_te[:, q] == 1)
            fids[arch].append(fid_pred(pq, Y_te[:, q]))

    base = corr["bx_single"].mean(1)
    print(f"{'Architecture':<12s} {'BRAM/Q':>6s} {'#Mult/Q':>7s} | {'Mean F':>7s} | {'Gain vs bx_single (pts)':>23s}")
    print("-" * 66)
    for arch in ARCHS:
        bram = 1 if not arch.startswith("df") else 2
        d = corr[arch].mean(1) - base
        print(f"{arch:<12s} {bram:6d} {n_mults[arch]:7d} | {np.mean(fids[arch]):7.4f} | "
              f"{100 * d.mean():+10.2f} +/- {100 * d.std() / np.sqrt(len(d)):.2f}")

    d_lin = corr["sf_lin20"].mean(1) - corr["df_lin20"].mean(1)
    d_cr  = corr["sf_cross"].mean(1) - corr["df_cross"].mean(1)
    print(f"  -> New Split-Latch vs Old Dual-Filter (Linear 20): {100*d_lin.mean():+.3f} +/- {100*d_lin.std()/np.sqrt(len(d_lin)):.3f} pts (saves 50% BRAM & MACs!)")
    print(f"  -> New Split-Latch vs Old Dual-Filter (Cross  47): {100*d_cr.mean():+.3f} +/- {100*d_cr.std()/np.sqrt(len(d_cr)):.3f} pts (saves 50% BRAM & MACs!)")

loading source-shot pools ...

################ PART 1: 5-Qubit Bus (Boxcar vs Old Dual-Filter vs New Split-Latch) ################

=== Crosstalk strength s = 0.0 (Boxcar window [75, 500), Split-Latch T_mid = 225) ===
Architecture BRAM/Q #Mult/Q |  Mean F | Gain vs bx_single (pts)
------------------------------------------------------------------
bx_single         1       3 |  0.8784 |      +0.00 +/- 0.00
bx_lin10          1      11 |  0.8784 |      -0.01 +/- 0.01
bx_cross          1      30 |  0.8784 |      -0.00 +/- 0.01
df_single         2       5 |  0.8819 |      +0.34 +/- 0.02
df_lin20          2      21 |  0.8819 |      +0.34 +/- 0.02
df_cross          2      47 |  0.8817 |      +0.33 +/- 0.02
sf_single         1       5 |  0.8819 |      +0.35 +/- 0.02
sf_lin20          1      21 |  0.8817 |      +0.32 +/- 0.02
sf_self30         1      31 |  0.8817 |      +0.32 +/- 0.02
sf_cross          1      47 |  0.8816 |      +0.32 +/- 0.02
  -> New Split-Latch vs Old Dual-Filter (Linear 20